# DNA Promoter Sequence Classification with Bidirectional LSTM & Recurrent Neural Networks

**Author:** Azheem Syahmee (B23CS0031)  
**Course:** SECB 3223 — Computational Biology 2, Universiti Teknologi Malaysia (UTM)  
**Dataset:** Molecular Biology (Promoter Gene Sequences), UCI Machine Learning Repository  

---

## 1. Biological Problem Overview

In prokaryotic genomics (*Escherichia coli*), **promoters** are specific DNA sequence motifs upstream of transcription start sites (TSS) that recruit RNA polymerase holoenzyme (specifically the $\sigma^{70}$ subunit) to initiate gene transcription.

Canonical *E. coli* $\sigma^{70}$ promoters contain two conserved hexameric consensus regions:
1. **-35 Region:** Typically `TTGACA`, located ~35 base pairs upstream of TSS.
2. **-10 Region (Pribnow Box):** Typically `TATAAT`, located ~10 base pairs upstream of TSS.
3. **Spacer:** 16–18 base pairs separating the -35 and -10 motifs.

Predicting whether an arbitrary 57-bp sequence possesses promoter functionality requires capturing non-local sequential dependencies across nucleotides. This study evaluates five distinct recurrent neural network architectures, highlighting **Bidirectional LSTMs** for bidirectional genomic context extraction.

In [ ]:
# ==============================================================================
# STEP 1: IMPORT REQUIRED LIBRARIES & SET RANDOM SEEDS
# ==============================================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional, LeakyReLU
from tensorflow.keras.callbacks import EarlyStopping

np.random.seed(42)
tf.random.set_seed(42)
print("TensorFlow Version:", tf.__version__)


In [ ]:
# ==============================================================================
# STEP 2: LOAD PROMOTER DATASET (WITH LOCAL & ONLINE RESILIENCE)
# ==============================================================================
local_paths = [
    os.path.join("..", "data", "promoters.data"),
    os.path.join("data", "promoters.data"),
    "promoters.data"
]
data_path = next((p for p in local_paths if os.path.exists(p)), None)

if data_path:
    print(f"Loading local dataset from: {data_path}")
    df = pd.read_csv(data_path, header=None, names=["label", "id", "sequence"])
else:
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/molecular-biology/promoter-gene-sequences/promoters.data"
    print(f"Fetching dataset from UCI repository: {url}")
    df = pd.read_csv(url, header=None, names=["label", "id", "sequence"])

print(f"Total instances loaded: {len(df)}")
print("\nClass Distribution:")
print(df["label"].value_counts())
df.head()


In [ ]:
# ==============================================================================
# STEP 3: SANITIZATION & ONE-HOT SEQUENCE ENCODING
# ==============================================================================
# Clean sequence whitespace & lowercase
df["sequence"] = df["sequence"].str.replace(r"\s+", "", regex=True).str.lower()
valid_bases = set("acgt")
df = df[df["sequence"].apply(lambda s: set(s).issubset(valid_bases))].copy()

# Binary label mapping: '+' -> 1 (Promoter), '-' -> 0 (Non-promoter)
df["target"] = df["label"].map({"+": 1, "-": 0})

base_to_vector = {
    "a": [1, 0, 0, 0],
    "c": [0, 1, 0, 0],
    "g": [0, 0, 1, 0],
    "t": [0, 0, 0, 1]
}

def encode_seq(sequence):
    return np.array([base_to_vector[b] for b in sequence], dtype=np.float32)

X = np.stack(df["sequence"].apply(encode_seq).values)
y = df["target"].values.astype(np.float32)

# Stratified 80/20 train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Feature Tensor Shape: {X.shape} (N={X.shape[0]}, L={X.shape[1]} bp, D=4 bases)")
print(f"Training set: {X_train.shape[0]} samples (Pos: {int(np.sum(y_train==1))}, Neg: {int(np.sum(y_train==0))})")
print(f"Test set    : {X_test.shape[0]} samples (Pos: {int(np.sum(y_test==1))}, Neg: {int(np.sum(y_test==0))})")


## 2. Experiment 1: Bio-Aware Bidirectional LSTM (Primary Architecture)

Customized using student ID parameterization (**B23CS0031**): `LSTM(31)` and `RMSprop(learning_rate=0.0031)`.

In [ ]:
model_exp1 = Sequential([
    Bidirectional(LSTM(31, return_sequences=False), input_shape=(57, 4)),
    Dropout(0.3),
    Dense(32, activation="tanh"),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
], name="Exp1_BioAware_BiLSTM")

model_exp1.compile(
    optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.0031),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stop = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)

history_exp1 = model_exp1.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=16,
    callbacks=[early_stop],
    verbose=1
)

loss_1, acc_1 = model_exp1.evaluate(X_test, y_test, verbose=0)
print(f"[EXP 1] Test Loss: {loss_1:.4f} | Test Accuracy: {acc_1:.4f}")


## 3. Experiment 2: Deep Memory Stacked LSTM

Two cascaded LSTM layers with LeakyReLU and Adamax optimization.

In [ ]:
model_exp2 = Sequential([
    LSTM(31, return_sequences=True, input_shape=(57, 4)),
    LSTM(31),
    Dropout(0.5),
    Dense(32),
    LeakyReLU(),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
], name="Exp2_DeepMemory_StackedLSTM")

model_exp2.compile(
    optimizer=tf.keras.optimizers.Adamax(learning_rate=0.0031),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history_exp2 = model_exp2.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=16,
    callbacks=[early_stop],
    verbose=0
)

loss_2, acc_2 = model_exp2.evaluate(X_test, y_test, verbose=0)
print(f"[EXP 2] Test Loss: {loss_2:.4f} | Test Accuracy: {acc_2:.4f}")


## 4. Evaluation & Diagnostic Summary

Confusion matrix and detailed classification report on the holdout test set.

In [ ]:
preds_prob = model_exp1.predict(X_test)
preds = (preds_prob >= 0.5).astype(int).flatten()

cm = confusion_matrix(y_test, preds)
print("Confusion Matrix (Exp 1 Bi-LSTM):")
print(cm)
print("\nClassification Report:")
print(classification_report(y_test, preds, target_names=["Non-Promoter", "Promoter"]))
